In [ ]:
#| default_exp solubility_batch

# Table-attached solubility batch (LangGraph + mat-gram)

Runs on **in-memory rows** from a Graph-tab table attached in chat.

LangGraph flow:

1. **analyze** — pandas summary + SMILES column candidates
2. **plan** — pick SMILES input column + prediction output column (Lisette, with heuristic fallback)
3. **predict** — POST unique SMILES to mat-gram `/embeddings`
4. **join** — pandas/numpy merge predictions back onto every row

Serve for the AGE router with:

```bash
pixi run serve-solubility-batch
```

`POST /solubility/batch` body: `{ "message": "...", "table": { "rows": [...] } }`.

In [ ]:
#| export
from __future__ import annotations

import json
import math
import os
import re
from contextvars import ContextVar
from functools import lru_cache
from typing import Any, TypedDict

import httpx
import numpy as np
import pandas as pd
from langgraph.graph import END, START, StateGraph
from lisette import Chat, contents

DEFAULT_MATGRAM_URL = "http://127.0.0.1:8080"
DEFAULT_TIMEOUT = float(os.environ.get("MATGRAM_TIMEOUT", "120"))
BATCH_PORT = int(os.getenv("BATCH_PORT", "8014"))
BATCH_HOST = os.getenv("BATCH_HOST", "0.0.0.0")

_SMILES_COL_NAMES = ("smiles", "smiles_pattern", "canonical_smiles", "smi")
_SMILES_TOKEN = re.compile(
    r"^(?:Br|Cl|Si|Se|Na|Li|Mg|Zn|Fe|Cu|B|C|N|O|P|S|F|I|H|"
    r"[cnops]|[=#+\-\(\)\[\]\\/@%0-9]){2,}$"
)

_current_df: ContextVar[pd.DataFrame | None] = ContextVar("solubility_batch_df", default=None)


def matgram_base() -> str:
    return (os.environ.get("MATGRAM_API_URL") or DEFAULT_MATGRAM_URL).rstrip("/")


def content_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, list):
        return "".join(
            (x.get("text") or "") if isinstance(x, dict) else str(x) for x in content
        )
    return str(content)


def asdict(m: Any) -> dict:
    if isinstance(m, dict):
        return m
    if hasattr(m, "model_dump"):
        return m.model_dump()
    try:
        return dict(m)
    except Exception as ex:  # noqa: BLE001
        raise TypeError(f"cannot convert {type(m)} to dict") from ex


def get_lisette_llm_config() -> dict[str, str]:
    model = (os.getenv("LISETTE_MODEL") or "").strip()
    key = (os.getenv("LISETTE_API_KEY") or "").strip() or (
        os.getenv("OPENAI_API_KEY") or ""
    ).strip()
    base = (os.getenv("LISETTE_API_BASE") or "").strip()
    if not model:
        return {}
    out: dict[str, str] = {"model": model}
    if key:
        out["api_key"] = key
    if base:
        out["api_base"] = base
    return out


def _active_df() -> pd.DataFrame:
    df = _current_df.get()
    if df is None:
        raise ValueError("No dataframe bound for solubility batch run")
    return df


def _jsonable(obj: Any) -> Any:
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return _jsonable(obj.item())
    if pd.isna(obj):
        return None
    item = getattr(obj, "item", None)
    if callable(item):
        try:
            return _jsonable(item())
        except Exception:  # noqa: BLE001
            pass
    return str(obj)


def _rows_from_table(table: Any) -> list[dict[str, Any]]:
    """Extract row dicts from a ``table`` hold payload (list or ``{rows: [...]}``)."""
    if isinstance(table, list):
        return [r for r in table if isinstance(r, dict)]
    if isinstance(table, dict):
        rows = table.get("rows")
        if isinstance(rows, list):
            return [r for r in rows if isinstance(r, dict)]
    return []


def _looks_like_smiles(val: Any) -> bool:
    if not isinstance(val, str):
        return False
    s = val.strip()
    if len(s) < 2 or " " in s:
        return False
    if not any(c.isalpha() for c in s):
        return False
    return bool(_SMILES_TOKEN.match(s))


def detect_smiles_column(df: pd.DataFrame) -> str | None:
    """Pick a SMILES column by known names, else by value heuristics."""
    if df.empty:
        return None
    cols = [str(c) for c in df.columns]
    by_lower = {c.lower(): c for c in cols}
    for name in _SMILES_COL_NAMES:
        if name in by_lower:
            return by_lower[name]
    sample = df.head(min(12, len(df)))
    best_col: str | None = None
    best_hits = 0
    for c in cols:
        hits = int(sample[c].map(_looks_like_smiles).sum())
        if hits > best_hits:
            best_hits = hits
            best_col = c
    if best_col is not None and best_hits >= max(1, len(sample) // 2):
        return best_col
    return None


def parse_pred_column(message: str) -> str:
    """Pick output column name from the user message; default ``esol``."""
    text = (message or "").strip()
    if not text:
        return "esol"
    patterns = [
        r"(?:new\s+)?column\s+named\s+[\"']?([A-Za-z_][\w]*)",
        r"(?:named|called)\s+[\"']?([A-Za-z_][\w]*)",
        r"\binto\s+(?:column\s+)?[\"']?([A-Za-z_][\w]*)",
        r"\bas\s+[\"']?([A-Za-z_][\w]*)",
        # "new pred column" / "a pred column with the result"
        r"\b(?:a|the|new)\s+([A-Za-z_][\w]*)\s+column\b",
    ]
    skip = {
        "smiles", "the", "a", "an", "column", "results", "result",
        "new", "output", "prediction", "predictions",
    }
    for pat in patterns:
        m = re.search(pat, text, re.I)
        if m:
            name = m.group(1)
            if name.lower() not in skip:
                return name
    return "esol"


def call_matgram_embeddings(
    smiles: list[str],
    *,
    base_url: str | None = None,
    timeout: float = DEFAULT_TIMEOUT,
) -> dict[str, Any]:
    """POST ``{smiles: [...]}`` to mat-gram ``/embeddings``."""
    url = f"{(base_url or matgram_base())}/embeddings"
    with httpx.Client(timeout=timeout) as client:
        r = client.post(url, json={"smiles": smiles})
        try:
            data = r.json()
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError(f"mat-gram returned non-JSON ({r.status_code})") from exc
        if r.status_code >= 400:
            detail = data.get("error") if isinstance(data, dict) else data
            raise RuntimeError(f"mat-gram HTTP {r.status_code}: {detail}")
        if not isinstance(data, dict):
            raise RuntimeError("mat-gram response must be a JSON object")
        return data


In [ ]:
#| export
class BatchState(TypedDict, total=False):
    question: str
    columns: str
    candidates: str
    result: str
    smiles_column: str
    pred_column: str
    predictions: list[float | None]
    unique_smiles: list[str]
    rows: list[dict[str, Any]] | None
    matgram: dict[str, Any]
    error: str


def analyze_table(state: BatchState) -> dict:
    """Summarize the attached table and list SMILES column candidates."""
    df = _active_df()
    cols = [str(c) for c in df.columns]
    heuristic = detect_smiles_column(df)
    candidates: list[str] = []
    by_lower = {c.lower(): c for c in cols}
    for name in _SMILES_COL_NAMES:
        if name in by_lower and by_lower[name] not in candidates:
            candidates.append(by_lower[name])
    if heuristic and heuristic not in candidates:
        candidates.append(heuristic)
    # Also include object columns with high SMILES hit rate
    sample = df.head(min(12, len(df)))
    for c in cols:
        if c in candidates:
            continue
        hits = int(sample[c].map(_looks_like_smiles).sum()) if len(sample) else 0
        if hits >= max(1, len(sample) // 2):
            candidates.append(c)

    summary = (
        f"shape={df.shape[0]} rows × {df.shape[1]} cols\n"
        f"dtypes:\n{df.dtypes.to_string()}\n\n"
        f"head:\n{df.head(8).to_string()}"
    )
    result = (
        f"Question: {state.get('question', '')}\n\n"
        f"Data summary:\n{summary}\n\n"
        f"SMILES column candidates: {', '.join(candidates) or '(none)'}"
    )
    return {
        "result": result,
        "columns": ",".join(cols),
        "candidates": ",".join(candidates),
        "smiles_column": heuristic or "",
        "pred_column": parse_pred_column(state.get("question", "") or ""),
    }


def plan_columns(state: BatchState) -> dict:
    """Choose SMILES + prediction column names.

    Prefer deterministic heuristics (fast/reliable). Only call Lisette when no
    SMILES candidate was found, or when ``SOLUBILITY_BATCH_LLM_PLAN=1``.
    """
    cols = [c for c in (state.get("columns") or "").split(",") if c]
    candidates = [c for c in (state.get("candidates") or "").split(",") if c]
    smiles_fallback = state.get("smiles_column") or (candidates[0] if candidates else "")
    pred_fallback = state.get("pred_column") or parse_pred_column(state.get("question", "") or "")

    use_llm = (os.getenv("SOLUBILITY_BATCH_LLM_PLAN") or "").strip().lower() in {
        "1", "true", "yes", "on",
    }
    if smiles_fallback and not use_llm:
        return {
            "smiles_column": smiles_fallback,
            "pred_column": pred_fallback,
            "result": (state.get("result", "") or "")
            + f"\n\nPlan (heuristic): smiles={smiles_fallback!r}, pred={pred_fallback!r}.",
        }

    cfg = get_lisette_llm_config()
    model = cfg.get("model")
    if not model or not cols:
        return {
            "smiles_column": smiles_fallback,
            "pred_column": pred_fallback,
            "result": (state.get("result", "") or "")
            + f"\n\nPlan (heuristic): smiles={smiles_fallback!r}, pred={pred_fallback!r}.",
        }

    prompt = f"""You route a table batch ESOL prediction.

Question: {state.get('question', '')}

All columns: {', '.join(cols)}
SMILES candidates: {', '.join(candidates) or '(none)'}

Return ONLY compact JSON (no markdown) with keys:
  "smiles_column": exact column name containing SMILES strings
  "pred_column": name for the new prediction column (e.g. pred, esol)

Rules:
- smiles_column MUST be one of the listed columns (prefer candidates).
- pred_column must be a valid Python identifier; default "esol" if unclear.
- If the user asks for a column named pred/score/etc, use that name.
"""
    try:
        c = Chat(
            model,
            temp=0,
            api_key=cfg.get("api_key"),
            api_base=cfg.get("api_base"),
        )
        raw_res = c(prompt, max_steps=2, max_tokens=400, return_all=False, stream=False)
        final = contents(raw_res)
        text = content_text(asdict(final).get("content")) if final else ""
        m = re.search(r"\{.*\}", text, re.DOTALL)
        data = json.loads(m.group(0) if m else text)
        smiles_col = str(data.get("smiles_column") or "").strip()
        pred_col = str(data.get("pred_column") or "").strip()
        if smiles_col not in cols:
            smiles_col = smiles_fallback
        if not re.match(r"^[A-Za-z_][\w]*$", pred_col or ""):
            pred_col = pred_fallback
        return {
            "smiles_column": smiles_col,
            "pred_column": pred_col,
            "result": (state.get("result", "") or "")
            + f"\n\nPlan: smiles={smiles_col!r}, pred={pred_col!r}.",
        }
    except Exception as e:  # noqa: BLE001
        return {
            "smiles_column": smiles_fallback,
            "pred_column": pred_fallback,
            "result": (state.get("result", "") or "")
            + f"\n\nPlan (fallback after LLM error {e}): smiles={smiles_fallback!r}, pred={pred_fallback!r}.",
        }


def predict_unique(state: BatchState) -> dict:
    """Call mat-gram once per unique non-empty SMILES (pandas unique)."""
    smiles_col = (state.get("smiles_column") or "").strip()
    if not smiles_col:
        return {
            "error": "No SMILES column selected.",
            "result": (state.get("result", "") or "") + "\n\nNo SMILES column selected.",
            "unique_smiles": [],
            "predictions": [],
        }

    df = _active_df()
    if smiles_col not in df.columns:
        return {
            "error": f"Column {smiles_col!r} not in table.",
            "result": (state.get("result", "") or "") + f"\n\nColumn {smiles_col!r} not in table.",
            "unique_smiles": [],
            "predictions": [],
        }

    series = df[smiles_col].astype("string").str.strip()
    series = series.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    unique = [s for s in series.dropna().unique().tolist() if isinstance(s, str) and s]
    if not unique:
        return {
            "error": f"Column {smiles_col!r} has no usable SMILES values.",
            "result": (state.get("result", "") or "")
            + f"\n\nColumn {smiles_col!r} has no usable SMILES values.",
            "unique_smiles": [],
            "predictions": [],
        }

    try:
        payload = call_matgram_embeddings(unique)
    except Exception as e:  # noqa: BLE001
        return {
            "error": str(e),
            "result": (state.get("result", "") or "") + f"\n\nmat-gram error: {e}",
            "unique_smiles": unique,
            "predictions": [],
            "matgram": {},
        }

    preds_raw = payload.get("predictions")
    if not isinstance(preds_raw, list) or len(preds_raw) != len(unique):
        return {
            "error": "mat-gram did not return matching predictions.",
            "result": (state.get("result", "") or "")
            + "\n\nmat-gram did not return ESOL predictions (check property/esol mode).",
            "unique_smiles": unique,
            "predictions": [],
            "matgram": {
                "mode": payload.get("mode"),
                "task": payload.get("task"),
                "model": payload.get("model"),
            },
        }

    preds: list[float | None] = []
    for p in preds_raw:
        try:
            preds.append(float(p) if p is not None and not (isinstance(p, float) and np.isnan(p)) else None)
        except (TypeError, ValueError):
            preds.append(None)

    return {
        "unique_smiles": unique,
        "predictions": preds,
        "matgram": {
            "mode": payload.get("mode"),
            "task": payload.get("task"),
            "model": payload.get("model"),
        },
        "result": (state.get("result", "") or "")
        + f"\n\nPredicted {len(unique)} unique SMILES via mat-gram.",
        "error": "",
    }


def join_predictions(state: BatchState) -> dict:
    """Map unique-SMILES predictions onto all rows with pandas merge."""
    if state.get("error"):
        return {"rows": None, "result": state.get("result", "")}

    smiles_col = (state.get("smiles_column") or "").strip()
    pred_col = (state.get("pred_column") or "esol").strip() or "esol"
    unique = state.get("unique_smiles") or []
    preds = state.get("predictions") or []
    if not unique or len(unique) != len(preds):
        return {
            "rows": None,
            "error": "No predictions to join.",
            "result": (state.get("result", "") or "") + "\n\nNo predictions to join.",
        }

    df = _active_df().copy()
    # Normalize join key the same way as predict_unique
    key = "_st_smiles_key"
    df[key] = df[smiles_col].astype("string").str.strip()
    df[key] = df[key].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    lookup = pd.DataFrame({key: unique, pred_col: np.asarray(preds, dtype=object)})
    # Coerce numeric preds; keep None as NA
    lookup[pred_col] = pd.to_numeric(lookup[pred_col], errors="coerce")

    if pred_col in df.columns and pred_col != smiles_col:
        df = df.drop(columns=[pred_col])

    out = df.merge(lookup, on=key, how="left")
    out = out.drop(columns=[key])

    records = out.to_dict(orient="records")
    clean = [_jsonable(r) if isinstance(r, dict) else {} for r in records]
    n_ok = int(out[pred_col].notna().sum())
    return {
        "rows": clean,
        "pred_column": pred_col,
        "result": (state.get("result", "") or "")
        + f"\n\nJoined predictions: {n_ok}/{len(out)} rows → column {pred_col!r}.",
        "error": "",
    }

In [ ]:
#| export
def build_graph():
    graph = StateGraph(BatchState)
    graph.add_node("analyze", analyze_table)
    graph.add_node("plan", plan_columns)
    graph.add_node("predict", predict_unique)
    graph.add_node("join", join_predictions)
    graph.add_edge(START, "analyze")
    graph.add_edge("analyze", "plan")
    graph.add_edge("plan", "predict")
    graph.add_edge("predict", "join")
    graph.add_edge("join", END)
    return graph.compile()


@lru_cache(maxsize=1)
def get_graph():
    return build_graph()


def run_batch(
    rows: list[dict[str, Any]],
    *,
    columns: list[str] | None = None,
    message: str = "",
) -> dict[str, Any]:
    """Run the LangGraph solubility batch on in-memory table rows."""
    if not rows:
        return {
            "status": "error",
            "error": "No table rows to predict.",
            "response": "No table rows to predict.",
            "source": "solubility_batch",
        }

    df = pd.DataFrame(rows)
    if columns:
        # Preserve declared column order when present
        ordered = [c for c in columns if c in df.columns]
        rest = [c for c in df.columns if c not in ordered]
        if ordered:
            df = df[ordered + rest]

    token = _current_df.set(df)
    try:
        out = get_graph().invoke(
            {
                "question": (message or "").strip(),
                "columns": ",".join(map(str, df.columns.tolist())),
                "candidates": "",
                "result": "",
                "smiles_column": "",
                "pred_column": "",
                "predictions": [],
                "unique_smiles": [],
                "rows": None,
                "matgram": {},
                "error": "",
            }
        )
    except Exception as e:  # noqa: BLE001
        return {
            "status": "error",
            "error": str(e),
            "response": f"Solubility batch failed: {e}",
            "source": "solubility_batch",
        }
    finally:
        _current_df.reset(token)

    out_rows = out.get("rows") if isinstance(out, dict) else None
    response = (out.get("result") if isinstance(out, dict) else None) or ""
    err = (out.get("error") if isinstance(out, dict) else None) or ""
    smiles_col = (out.get("smiles_column") if isinstance(out, dict) else None) or ""
    pred_col = (out.get("pred_column") if isinstance(out, dict) else None) or "esol"
    matgram = (out.get("matgram") if isinstance(out, dict) else None) or {}

    if isinstance(out_rows, list) and out_rows and isinstance(out_rows[0], dict):
        n_ok = sum(1 for r in out_rows if r.get(pred_col) is not None)
        return {
            "status": "success",
            "response": (
                f"Predicted ESOL for {n_ok}/{len(out_rows)} molecules "
                f"(SMILES column {smiles_col} → {pred_col}, log10 mol/L)."
            ),
            "twin_name": "solubility_batch",
            "row_count": len(out_rows),
            "rows": out_rows,
            "smiles_column": smiles_col,
            "pred_column": pred_col,
            "source": "solubility_batch",
            "matgram": matgram,
            "detail": response,
        }

    return {
        "status": "error",
        "error": err or "No transformed table produced.",
        "response": response or err or "No transformed table produced.",
        "smiles_column": smiles_col,
        "pred_column": pred_col,
        "source": "solubility_batch",
        "matgram": matgram,
        "hint": "Attach a table with a SMILES column and ask to predict into a named column (e.g. pred).",
    }

## HTTP serve

Starlette app for the controller forward target (`hold: table`).

In [ ]:
#| export
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route


async def batch_endpoint(request: Request) -> JSONResponse:
    """POST /solubility/batch — LangGraph ESOL batch for attached table rows."""
    try:
        body = await request.json()
    except Exception:  # noqa: BLE001
        return JSONResponse({"status": "error", "detail": "Invalid JSON"}, status_code=400)
    if not isinstance(body, dict):
        return JSONResponse({"status": "error", "detail": "JSON object required"}, status_code=400)

    message = body.get("message") or body.get("user_message") or ""
    if not isinstance(message, str):
        message = str(message or "")

    table = body.get("table")
    rows = _rows_from_table(table)
    columns: list[str] | None = None
    if isinstance(table, dict):
        cols = table.get("columns")
        if isinstance(cols, list) and cols:
            columns = [str(c) for c in cols]

    if not rows:
        return JSONResponse(
            {
                "status": "error",
                "error": "Missing or empty 'table' (expected rows).",
                "response": "Missing or empty 'table' (expected rows).",
                "source": "solubility_batch",
            },
            status_code=400,
        )

    result = run_batch(rows, columns=columns, message=message.strip())
    status = 200 if isinstance(result, dict) and result.get("status") == "success" else 502
    return JSONResponse(result if isinstance(result, dict) else {"status": "error"}, status_code=status)


async def health(_: Request) -> JSONResponse:
    """GET /health — liveness for deploy checks."""
    return JSONResponse({"status": "ok", "service": "solubility_batch"})


app = Starlette(
    routes=[
        Route("/solubility/batch", batch_endpoint, methods=["POST"]),
        Route("/health", health, methods=["GET"]),
    ]
)


if __name__ == "__main__":
    import uvicorn

    print(f"Serving solubility batch on http://{BATCH_HOST}:{BATCH_PORT}/solubility/batch")
    uvicorn.run(app, host=BATCH_HOST, port=BATCH_PORT)

In [ ]:
# Demo (not exported): requires mat-gram on :8080 in esol property mode
# rows = [
#     {"zinc_id": "ZINC1", "smiles": "CCO"},
#     {"zinc_id": "ZINC2", "smiles": "c1ccccc1"},
#     {"zinc_id": "ZINC3", "smiles": "CCO"},  # duplicate → same pred via merge
# ]
# run_batch(rows, message="predict into column pred")